# Four-model depth and pose comparison

PanoVGGT · VGGT-360 · OmniVGGT · MapAnything

This is an image-only qualitative experiment. GT investigation is paused; no GT poses or depth are passed to any model. Follow `docs/model_comparison.md` to install the separate model environments and download pinned source code first.

**GPU integration status:** runners have been checked against upstream APIs, but pretrained GPU inference has not been executed in the authoring environment. The first server run is a smoke test.

OmniVGGT and MapAnything use our six-perspective-view adaptation. VGGT-360 processes panoramas independently using its official fusion. These are different inference protocols, not a controlled accuracy benchmark.

In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
SEQUENCE = ROOT / 'standard_GT' / 'city01_route01'
JOB = ROOT / 'outputs' / 'depth_comparison' / 'run01'
FRAME_IDS = [0, 20]  # Retained frame IDs, shared by every model. Increase after the smoke test.
ENVIRONMENTS = {
    'panovggt': None,  # None uses this notebook's Python. Set your existing conda env name if different.
    'vggt360': 'vggt360',
    'omnivggt': 'omnivggt',
    'mapanything': 'mapanything',
}
CHECKPOINTS = {
    'panovggt': '/absolute/path/to/your/panovggt/model.pt',  # EDIT THIS
    'omnivggt': None,  # None downloads the official safetensors checkpoint.
}
ENABLED_MODELS = ['panovggt', 'vggt360', 'omnivggt', 'mapanything']
# For a first run you can use ['vggt360'] and enable the others afterwards.


## Prepare the shared images

A new job copies the selected RGB images and creates six 100-degree perspective views per panorama. Existing jobs are reused only when the frame selection matches. No pose values are copied into the model inputs. To change frames or source sequence, choose a new JOB folder.

In [ ]:
from comparison.prepare import prepare_job
if not (JOB / 'manifest.json').exists():
    prepare_job(SEQUENCE, JOB, FRAME_IDS)
manifest = json.loads((JOB / 'manifest.json').read_text())
assert manifest['source_sequence'] == str(SEQUENCE.resolve()), 'Use a new JOB for a different source sequence.'
assert [r['frame_id'] for r in manifest['frames']] == FRAME_IDS, 'Use a new JOB for a different selection.'
import pandas as pd
from IPython.display import display
display(pd.DataFrame(manifest['frames'])[['frame_id', 'source_frame_id', 'source_name']])
print('Job:', JOB)
print('Images per panorama: 1 ERP or 6 virtual perspective views.')


In [ ]:
def run_model(name):
    if name not in ENABLED_MODELS:
        print(f'{name}: disabled'); return
    target = JOB / 'predictions' / name
    if (target / 'metadata.json').exists():
        print(f'{name}: completed output already exists; reuse it for plotting.'); return
    if target.exists():
        print(f'{name}: incomplete run at {target}. Move it aside before retrying.'); return
    env = ENVIRONMENTS[name]
    if env is None:
        command = [sys.executable]
    else:
        conda = shutil.which('conda')
        if conda is None: raise RuntimeError('conda is not on PATH; open VS Code from a conda-enabled shell.')
        command = [conda, 'run', '--no-capture-output', '-n', env, 'python']
    command += ['-m', 'comparison.run', '--job', str(JOB), '--model', name]
    checkpoint = CHECKPOINTS.get(name)
    if checkpoint:
        if not Path(checkpoint).is_file():
            print(f'{name}: set an existing checkpoint path in CHECKPOINTS.'); return
        command += ['--checkpoint', str(checkpoint)]
    print('Running:', command, flush=True)
    # Stream logs into the notebook. Each model exits before the next one loads.
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
    print(f'\n{name}: exit code {status}')
    if status: print('Run failed; inspect the log above. No completed result will be shown.')


In [ ]:
run_model('panovggt')


In [ ]:
run_model('vggt360')


In [ ]:
run_model('omnivggt')


In [ ]:
run_model('mapanything')


## Outputs and inference metadata

VGGT-360 depth follows the official world-point-norm fusion; it is not guaranteed metric camera-centred distance. OmniVGGT z-depth is converted to radial distance with predicted intrinsics. MapAnything supplies depth along rays. The perspective models' maps are placed back into ERP with known crop geometry; inconsistent predicted calibration or face poses may produce seams.

No per-face rescaling or GT fitting is performed. Read each model's scale and depth-semantics fields before comparing raw values.

In [ ]:
from comparison.view import load_results, depth_grid, trajectories
manifest, results = load_results(JOB)
display(pd.DataFrame([{'model': name, **meta} for name, (_, meta) in results.items()]))


## Shape comparison (independent percentile normalization)

Each map uses its own 2nd/98th percentiles. This highlights structure but hides scale differences. It is not a depth accuracy score. Missing model outputs remain explicitly marked.

In [ ]:
import matplotlib.pyplot as plt
fig = depth_grid(JOB, normalized=True)
fig.savefig(JOB / 'depth_normalized.png', dpi=150, bbox_inches='tight')
plt.show()


## Raw model outputs

Each map has its own numerical colorbar by default. Use `raw_limits=(0, 100)` only when you intentionally want one numerical range; this does not make scales comparable or turn units into metres.

In [ ]:
fig = depth_grid(JOB, normalized=False, raw_limits=None)
fig.savefig(JOB / 'depth_raw.png', dpi=150, bbox_inches='tight')
plt.show()


## Predicted camera paths

Each path is expressed relative to its own first camera, in that model's scale. The front virtual view represents each ERP for the perspective models. No GT alignment or pose scoring is performed. Two frames give only a line segment; add more shared frames in a new job after successful smoke tests.

VGGT-360's per-panorama virtual-view poses are saved separately and are **not a temporal trajectory**.

In [ ]:
fig = trajectories(JOB)
if fig is not None:
    fig.savefig(JOB / 'predicted_paths.png', dpi=150, bbox_inches='tight')
    plt.show()


## Share the results

The compact report contains the manifest, metadata and plots, not checkpoints or source images. NPZ predictions stay in the job folder for later analysis.

In [ ]:
import zipfile
report_path = JOB / 'comparison_report.zip'
with zipfile.ZipFile(report_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    candidates = [JOB / 'manifest.json', *JOB.glob('*.png'), *JOB.glob('predictions/*/metadata.json')]
    for path in candidates:
        archive.write(path, path.relative_to(JOB))
print('Share this report:', report_path)
